In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers.advanced_activations import *
from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
def fraction_within_eps(y_true, y_pred, epsilon=0.5):
    # fraction of entries where abs(y_true - y_pred) < epsilon
    
    count = np.sum(np.abs(y_true - y_pred) <= epsilon)
    return count / y_true.shape[0]

yt = np.array([10, 10, 10, 10, 10])
yp = np.array([9.2, 9.7, 10.3, 10.2, 11])
print(fraction_within_eps(yt, yp, 0.6))
FIE = fraction_within_eps

0.6


In [3]:
def concorr(x, y):
    # Return Lin's concordance correlation coefficient
    # x, y are numpy arrays
    
    xm = x.mean()
    ym = y.mean()
    xv = x.var()
    yv = y.var()
    xycov = np.sum((x-xm)*(y-ym)) / x.shape[0]
    lin = 2*xycov / (xv + yv + (xm - ym)**2)
    return lin

xx = np.random.randn(10)
yy = xx + 10
print('R2 corr coef is {} whereas concordance corr coef is {}'.format(R2(yy,xx), concorr(yy,xx)))

R2 corr coef is -141.79842289208372 whereas concordance corr coef is 0.013812305134639295


In [4]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


From the best 21 configurations modes of respective categories are as follows.

**Initializer:** normal

**Layers:** 2

**Batch Size:** 64

**Optimizer:** adamax

**Shuffle:** True

**Scaler:** RobustScaler

**Loss:** mean_absolute_error

In [5]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [6]:
def radstimator(h1=20, h2=15, num_vars=5):
    init = 'normal'
    
    model = Sequential()
    model.add( Dense( h1, kernel_initializer=init, input_dim=num_vars ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( BatchNorm())
    model.add( Dense( h2, kernel_initializer=init ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( Dropout( rate=0.4 ))
    
    model.add( Dense( 1, kernel_initializer=init, activation='linear' ))
    
    return model

In [9]:
print(x_train6_[:5]) # 'Latitude', 'BSH', 'Temperature(avg)', 'Daylength', 'H0'

[[40.141       5.9         4.22916667  9.19499685 13.69287119]
 [40.141       1.3         7.6375      9.20626964 13.74607822]
 [40.141       0.          6.2375      9.21860396 13.80432176]
 [40.141       0.          3.32916667  9.23198817 13.86758193]
 [40.141       0.7         5.06956522  9.24640976 13.93583675]]


In [7]:
data6_2v_ = {'x_train': x_train6_[:, [1, 3]], 'x_dev': x_dev_[:, [1, 3]], 'x_test': x_test_[:, [1, 3]]}
data_2v_ = {'x_train': x_train_[:, [1, 3]], 'x_dev': x_dev_[:, [1, 3]], 'x_test': x_test_[:, [1, 3]]}

data6_2v = scale_data(RobustScaler(), data6_2v_)
data_2v = scale_data(RobustScaler(), data_2v_)

x_train6_2v = data6_2v['x_train']
x_train_2v = data_2v['x_train']

x_dev6_2v = data6_2v['x_dev']
x_dev_2v = data_2v['x_dev']

x_test6_2v = data6_2v['x_test']
x_test_2v = data_2v['x_test']

y_train6_hh0 = y_train6 / x_train6_[:, 4]
y_train_hh0 = y_train / x_train_[:, 4]
y_dev_hh0 = y_dev / x_dev_[:, 4]
y_test_hh0 = y_test / x_test_[:, 4]

print(f'x_train shape: {x_train_2v.shape}, y_train shape: {y_train_hh0.shape}')
print(f'x_train6 shape: {x_train6_2v.shape}, y_train6 shape: {y_train6_hh0.shape}')
print(f'x_dev shape: {x_dev_2v.shape}, y_dev shape: {y_dev_hh0.shape}')
print(f'x_test shape: {x_test_2v.shape}, y_test shape: {y_test_hh0.shape}')

x_train shape: (52416, 2), y_train shape: (52416,)
x_train6 shape: (10654, 2), y_train6 shape: (10654,)
x_dev shape: (9011, 2), y_dev shape: (9011,)
x_test shape: (16899, 2), y_test shape: (16899,)


In [8]:
# Train with 6 stations data, and then with 31. Compare them.
# First for the variables n,N for H/H0.

validation_data6 = ( x_dev6_2v, y_dev_hh0 )
metrics_test6 = []
metrics_train6 = []
metrics_dev6 = []
for i in range(50):
    rads = radstimator(20, 15, 2)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-2vars/6stations-hh0-nN {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train6_2v, y_train6_hh0, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train6_2v, batch_size = 64 )

    mse = MSE( y_train6_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_train6_hh0, p )
    r2 = R2( y_train6_hh0, p )
    evs = EVS( y_train6_hh0, p )
    fie_h = FIE( y_train6_hh0, np.squeeze(p), 0.5)
    fie_o = FIE( y_train6_hh0, np.squeeze(p), 1)
    fie_oh = FIE( y_train6_hh0, np.squeeze(p), 1.5)
    lin = concorr(y_train6_hh0, np.squeeze(p))
    
    #metrics_train6.append((rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                            rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))
    """
    # Validation set
    p = rads.predict( x_dev6_2v, batch_size=64 )
    mse = MSE( y_dev_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_dev_hh0, p )
    r2 = R2( y_dev_hh0, p )
    evs = EVS( y_dev_hh0, p )
    fie_h = FIE( y_dev_hh0, np.squeeze(p), 0.5)
    fie_o = FIE( y_dev_hh0, np.squeeze(p), 1)
    fie_oh = FIE( y_dev_hh0, np.squeeze(p), 1.5)
    lin = concorr( y_dev_hh0, np.squeeze(p) )

    metrics_dev6.append((rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
          'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                        rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))
        
    # predict on test set and evaluate the metrics
    p = rads.predict( x_test6_2v, batch_size=64 )
    mse = MSE( y_test_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_test_hh0, p )
    r2 = R2( y_test_hh0, p )
    evs = EVS( y_test_hh0, p )
    fie_h = FIE( y_test_hh0, np.squeeze(p), 0.5)
    fie_o = FIE( y_test_hh0, np.squeeze(p), 1)
    fie_oh = FIE( y_test_hh0, np.squeeze(p), 1.5)
    lin = concorr(y_test_hh0, np.squeeze(p))

    metrics_test6.append((rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
          'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                        rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))"""

C01 » RMSE: 0.1360, MAE: 0.0761, R2: 0.6295, EVS: 0.6296, FIE_H: 0.9690, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7875
C02 » RMSE: 0.1360, MAE: 0.0776, R2: 0.6292, EVS: 0.6351, FIE_H: 0.9713, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7708
C03 » RMSE: 0.1356, MAE: 0.0759, R2: 0.6316, EVS: 0.6339, FIE_H: 0.9700, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7815
C04 » RMSE: 0.1362, MAE: 0.0772, R2: 0.6281, EVS: 0.6354, FIE_H: 0.9708, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7724
C05 » RMSE: 0.1375, MAE: 0.0759, R2: 0.6211, EVS: 0.6296, FIE_H: 0.9694, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7819
C06 » RMSE: 0.1369, MAE: 0.0765, R2: 0.6242, EVS: 0.6341, FIE_H: 0.9702, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7750
C07 » RMSE: 0.1365, MAE: 0.0751, R2: 0.6268, EVS: 0.6313, FIE_H: 0.9696, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7837
C08 » RMSE: 0.1351, MAE: 0.0756, R2: 0.6341, EVS: 0.6375, FIE_H: 0.9700, FIE_O: 0.9999, FIE_OH: 1.0000, LINCC: 0.7823
C09 » RMSE: 0.1366, MAE: 0.0745, R2: 0.6262, EVS: 0.6300

In [9]:
# Train with 6 stations data, and then with 31. Compare them.
# First for the variables n,N for H/H0.

validation_data = ( x_dev_2v, y_dev_hh0 )
metrics_test = []
metrics_train = []
metrics_dev = []
for i in range(50):
    rads = radstimator(20, 15, 2)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-2vars/31stations-hh0-nN {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train_2v, y_train_hh0, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train_2v, batch_size = 64 )

    mse = MSE( y_train_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_train_hh0, p )
    r2 = R2( y_train_hh0, p )
    evs = EVS( y_train_hh0, p )
    fie_h = FIE( y_train_hh0, np.squeeze(p), 0.5)
    fie_o = FIE( y_train_hh0, np.squeeze(p), 1)
    fie_oh = FIE( y_train_hh0, np.squeeze(p), 1.5)
    lin = concorr(y_train_hh0, np.squeeze(p))
    
    #metrics_train.append((rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                            rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))
    """
    # Validation set
    p = rads.predict( x_dev_2v, batch_size=64 )
    mse = MSE( y_dev_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_dev_hh0, p )
    r2 = R2( y_dev_hh0, p )
    evs = EVS( y_dev_hh0, p )
    fie_h = FIE( y_dev_hh0, np.squeeze(p), 0.5)
    fie_o = FIE( y_dev_hh0, np.squeeze(p), 1)
    fie_oh = FIE( y_dev_hh0, np.squeeze(p), 1.5)
    lin = concorr( y_dev_hh0, np.squeeze(p) )

    metrics_dev.append((rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
          'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                        rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))
        
    # predict on test set and evaluate the metrics
    p = rads.predict( x_test_2v, batch_size=64 )
    mse = MSE( y_test_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_test_hh0, p )
    r2 = R2( y_test_hh0, p )
    evs = EVS( y_test_hh0, p )
    fie_h = FIE( y_test_hh0, np.squeeze(p), 0.5)
    fie_o = FIE( y_test_hh0, np.squeeze(p), 1)
    fie_oh = FIE( y_test_hh0, np.squeeze(p), 1.5)
    lin = concorr(y_test_hh0, np.squeeze(p))

    metrics_test.append((rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
          'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                        rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))"""

C01 » RMSE: 0.1099, MAE: 0.0660, R2: 0.7033, EVS: 0.7063, FIE_H: 0.9879, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8166
C02 » RMSE: 0.1104, MAE: 0.0664, R2: 0.7006, EVS: 0.7035, FIE_H: 0.9881, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8140
C03 » RMSE: 0.1094, MAE: 0.0638, R2: 0.7056, EVS: 0.7077, FIE_H: 0.9868, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8289
C04 » RMSE: 0.1098, MAE: 0.0632, R2: 0.7037, EVS: 0.7079, FIE_H: 0.9863, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8338
C05 » RMSE: 0.1087, MAE: 0.0629, R2: 0.7094, EVS: 0.7096, FIE_H: 0.9865, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8352
C06 » RMSE: 0.1092, MAE: 0.0629, R2: 0.7072, EVS: 0.7074, FIE_H: 0.9861, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8351
C07 » RMSE: 0.1101, MAE: 0.0628, R2: 0.7024, EVS: 0.7024, FIE_H: 0.9853, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8394
C08 » RMSE: 0.1096, MAE: 0.0641, R2: 0.7050, EVS: 0.7071, FIE_H: 0.9867, FIE_O: 1.0000, FIE_OH: 1.0000, LINCC: 0.8267
C09 » RMSE: 0.1113, MAE: 0.0656, R2: 0.6957, EVS: 0.7013